# 05 · Modelo Early Fusion (mejor modelo)

**Objetivo:** Combinar imágenes y metadatos clínicos en un único modelo que aprenda las interacciones entre ambas modalidades de forma conjunta.

**Datos de entrada:** `../data/raw/hnmist_28_28_RGB.csv`, `../data/raw/HAM10000_metadata.csv`

**Resultado esperado:** Modelo guardado en `../models/mejor_modelo_early_fusion.keras` con ~80% de accuracy en test.

**Arquitectura:** Functional API con dos ramas (CNN + Dense) concatenadas antes de la capa de salida.

**Por qué Early Fusion supera a los modelos individuales:** al concatenar las representaciones de imagen y tabular antes de la clasificación final, el modelo puede aprender interacciones entre ambas fuentes (ej. una lesión pigmentada en la espalda de un hombre mayor tiene un perfil de riesgo diferente a la misma lesión en otra localización).

# Modelo 3D: Early Fusion
Ahora, vamos a utilizar la estrategia Early fusion para crear un modelo que combina los datos tabulares (el primer modelo que hicimos) con las imágenes para que el modelo tenga en cuenta además de los datos de imágenes, factores como en qué parte del cuerpo está la lesión, el sexo del paciente y su edad. En este caso, no tendríamos que importar todas las librerías que cargábamos antes, ya que vamos a reutilizar los modelos 1D y 2D 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, Flatten, Conv2D, MaxPooling2D, Concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

# ── Carga y alineamiento por image_id ────────────────────────────────────────
df_images = pd.read_csv(r"../data/raw/hnmist_28_28_RGB.csv")
metadata  = pd.read_csv(r"../data/raw/HAM10000_metadata.csv")

n = min(len(df_images), len(metadata))
df_images = df_images.iloc[:n].reset_index(drop=True)
metadata  = metadata.iloc[:n].reset_index(drop=True)
df_images['image_id'] = metadata['image_id']

df = df_images.merge(
    metadata[['image_id', 'dx', 'age', 'sex', 'localization']],
    on='image_id'
)

# ── Preprocesado de imágenes y tabular ────────────────────────────────────────
pixel_cols = [c for c in df.columns if c not in ['image_id', 'dx', 'age', 'sex', 'localization']]
X_img = df[pixel_cols].values.astype(np.float32).reshape(-1, 28, 28, 3)
X_img /= 255.0

tabular_data = pd.get_dummies(df[['age', 'sex', 'localization']], columns=['sex', 'localization'], drop_first=True)
X_tab = tabular_data.values.astype(np.float32)

le = LabelEncoder()
y_encoded = le.fit_transform(df['dx'].values)
y_onehot  = to_categorical(y_encoded)
num_classes = y_onehot.shape[1]
print("Clases:", le.classes_)

# ── Split 70 / 15 / 15 ───────────────────────────────────────────────────────
X_img_train, X_img_temp, X_tab_train, X_tab_temp, y_train, y_temp, enc_train, enc_temp = train_test_split(
    X_img, X_tab, y_onehot, y_encoded,
    test_size=0.30, random_state=42, stratify=y_encoded)
X_img_val, X_img_test, X_tab_val, X_tab_test, y_val, y_test = train_test_split(
    X_img_temp, X_tab_temp, y_temp,
    test_size=0.50, random_state=42, stratify=enc_temp)

print(f"Train: {X_img_train.shape[0]} | Val: {X_img_val.shape[0]} | Test: {X_img_test.shape[0]}")

# ── Imputación de NaNs: fit solo en train ────────────────────────────────────
X_tab_train = np.where(np.isinf(X_tab_train), np.nan, X_tab_train)
X_tab_val   = np.where(np.isinf(X_tab_val),   np.nan, X_tab_val)
X_tab_test  = np.where(np.isinf(X_tab_test),  np.nan, X_tab_test)

imp = SimpleImputer(strategy='median')
X_tab_train = imp.fit_transform(X_tab_train)
X_tab_val   = imp.transform(X_tab_val)
X_tab_test  = imp.transform(X_tab_test)

# ── Arquitectura Early Fusion ─────────────────────────────────────────────────
input_img = Input(shape=(28, 28, 3), name="input_img")
x = Conv2D(32,  (3,3), activation='relu', padding='same')(input_img)
x = MaxPooling2D((2,2))(x)
x = Conv2D(64,  (3,3), activation='relu', padding='same')(x)
x = MaxPooling2D((2,2))(x)
x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
x = MaxPooling2D((2,2))(x)
x = Flatten()(x)

input_tab = Input(shape=(X_tab_train.shape[1],), name="input_tab")
t = Dense(32, activation='relu')(input_tab)
t = Dense(32, activation='relu')(t)

combined = Concatenate()([x, t])
z = Dropout(0.5)(combined)
output = Dense(num_classes, activation='softmax')(z)

early_fusion_model = Model(inputs=[input_img, input_tab], outputs=output)
early_fusion_model.compile(optimizer=Adam(0.001),
                            loss='categorical_crossentropy',
                            metrics=['accuracy'])
early_fusion_model.summary()

# ── Entrenamiento ─────────────────────────────────────────────────────────────
early_stop = EarlyStopping(
    monitor='val_loss', patience=10, min_delta=0.005, restore_best_weights=True)

history = early_fusion_model.fit(
    [X_img_train, X_tab_train], y_train,
    validation_data=([X_img_val, X_tab_val], y_val),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop]
)

# ── Evaluación ────────────────────────────────────────────────────────────────
test_loss, test_acc = early_fusion_model.evaluate([X_img_test, X_tab_test], y_test)
print(f"Test accuracy: {test_acc:.4f}")

plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Accuracy — Early Fusion')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.show()


## Evaluación detallada por clase

El accuracy global no es suficiente en diagnóstico médico. Aquí vemos cuánto acierta el modelo en **cada tipo de lesión** por separado, y lo comparamos con el **baseline ZeroR** (lo que conseguiría un modelo que predice siempre la clase más frecuente, sin aprender nada).

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Predicciones sobre el conjunto de test
y_pred_probs = early_fusion_model.predict([X_img_test, X_tab_test])
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

# Baseline ZeroR: accuracy de predecir siempre la clase más frecuente
from collections import Counter
most_common = Counter(y_true).most_common(1)[0][1]
baseline = most_common / len(y_true)
print(f'Baseline ZeroR (predecir siempre la clase más frecuente): {baseline:.2%}')
print(f'Accuracy del modelo: {(y_pred == y_true).mean():.2%}')
print(f'Mejora sobre baseline: {(y_pred == y_true).mean() - baseline:+.2%}')

# Desglose por clase
print('
Resultados por clase diagnóstica:')
print(classification_report(y_true, y_pred, target_names=le.classes_))

# Modelo 3: Conclusiones 
En nuestro modelo 3 (Early fusion) Como vemos en la gráfica más arriba, el valor de accuracy ha estado aumentando durante todo el entrenamiento, lo cual nos deja conclusiones muy positivas para este modelo, ya que hemos llegado a alcanzar un 80% (valor más alto hasta ahora)
El modelo aprende de manera estable, y detiene su entrenamiento en epoch 23, el cual sería el punto donde detener el aprendizaje. 
Como vemos, combinar los datos tabulares con los resultados de la CNN ha conseguido que tengamos unas predicciones más fiables. Esto también nos indica que los datos tabualres pueden resultar de ayuda 
